In [ ]:
# ── INSTALAÇÃO AUTOMÁTICA DE DEPENDÊNCIAS ────────────────────────────────────
# Esta célula garante que todas as bibliotecas necessárias estão instaladas.
# Rode ela primeiro — ela detecta o que falta e instala automaticamente.

import subprocess, sys, importlib

PACOTES = {
    'pandas':      'pandas>=2.0',
    'numpy':       'numpy>=1.26',
    'matplotlib':  'matplotlib>=3.8',
    'yfinance':    'yfinance>=0.2.40',
    'pyarrow':     'pyarrow>=15.0',
    'scipy':       'scipy>=1.11',
    'statsmodels': 'statsmodels>=0.14',
    'sklearn':     'scikit-learn>=1.4',
    'anthropic':   'anthropic>=0.25',
}

instalou = False
for modulo, pacote in PACOTES.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        print(f"Instalando {pacote}...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacote],
                       check=False)
        instalou = True

if instalou:
    print("\n✓ Instalação concluída.")
    print("  Reiniciando o kernel para carregar as novas bibliotecas...")
    # Reinicia o kernel automaticamente (funciona no VS Code e no Colab)
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("  Reinicie o kernel manualmente (Ctrl+Shift+P → Restart Kernel)")
        print("  e rode todas as células novamente.")
else:
    print("✓ Todas as dependências já estão instaladas. Pode continuar.")

# Aula 9 — GenAI, Relatório & Defesa

Esta é a aula final do Intensivão Quant AI. Aqui vamos consolidar nossa jornada de duas formas:
1. **IA Generativa (Claude API):** para automatizar a redação técnica, análise crítica e formatação de relatórios.
2. **Relatório Técnico & Defesa Oral:** para estruturar o PDF final seguindo os 7 critérios do Itaú, gerar o Tear Sheet final e preparar a banca de defesa oral.

---

**Agenda:**
1. Como LLMs funcionam & Setup da API do Claude
2. Engenharia de prompt para análise financeira
3. Produção automatizada de seções do relatório
4. Crítica e refinamento multi-turno
5. Os 7 critérios de avaliação do Itaú
6. O Tear Sheet final (Painel de 5 gráficos)
7. Defesa oral: perguntas difíceis e rubrica
8. Checklist final de entrega

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Detecta se está rodando no VS Code ou no Google Colab e define DADOS_DIR —
# a pasta onde todos os arquivos de dados do curso serão salvos.
# Funciona em qualquer computador, sem precisar alterar nada.

import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente: Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Clonar o repositório do curso se ainda não existir
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run(['git', 'clone',
                        'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
                        REPO_DIR], check=False)

    # Dados salvos no Google Drive — persistem entre sessões
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente: VS Code / Jupyter local")

    # Sobe pastas até encontrar a raiz do repositório (onde fica o .git)
    # Funciona independente de onde o usuário salvou o projeto
    _p = os.path.abspath(os.getcwd())
    _root = None
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            _root = _p
            break
        _p = os.path.dirname(_p)

    if _root is None:
        # Fallback: usa a pasta onde o notebook está
        _root = os.path.dirname(os.path.abspath('__file__'))

    DADOS_DIR = os.path.join(_root, 'dados')

os.makedirs(DADOS_DIR, exist_ok=True)
print(f"Pasta de dados: {DADOS_DIR}")

In [ ]:
# ── VERIFICAÇÃO DE DADOS DA AULA ANTERIOR ────────────────────────────────────
_necessarios = [
    os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'),
    os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet'),
    os.path.join(DADOS_DIR, 'retorno_walkforward_liquido.parquet'),
]
_faltando = [f for f in _necessarios if not os.path.exists(f)]
if _faltando:
    print("\n⚠  Arquivos necessários não encontrados:")
    for f in _faltando:
        print(f"   • {os.path.basename(f)}")
    print(f"\n   Execute primeiro: aula-08-backtest-rigoroso")
    print(f"   Dados esperados em: {DADOS_DIR}")
else:
    print("✓  Dados encontrados. Pode continuar.")

In [ ]:
# Instalar dependências necessárias
# !pip install anthropic scipy matplotlib pandas numpy

import os, sys
import json
import textwrap
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from scipy import stats
import anthropic

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Configuração da API do Claude (lê ANTHROPIC_API_KEY automaticamente)
# Caso use localmente ou no Colab, certifique-se de configurar a API key
try:
    client = anthropic.Anthropic()  # lê da variável de ambiente
    print('SDK Anthropic configurado com sucesso.')
    print(f'Versão: {anthropic.__version__}')
except Exception as e:
    print('Aviso: ANTHROPIC_API_KEY não encontrada. As células de chamada do Claude falharão se não configurar a chave.')


In [ ]:
# ── VERIFICAÇÃO DE DEPENDÊNCIAS ──────────────────────────────────────────────
# Este notebook depende dos dados gerados pela aula anterior.
# Se algum arquivo estiver faltando, rode o notebook indicado primeiro.

_arquivos_necessarios = [
    os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'),
    os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet'),
    os.path.join(DADOS_DIR, 'retorno_walkforward_liquido.parquet'),
]

_faltando = [f for f in _arquivos_necessarios if not os.path.exists(f)]
if _faltando:
    print("\n⚠  ATENÇÃO: arquivos necessários não encontrados:")
    for f in _faltando:
        print(f"   Faltando: {os.path.basename(f)}")
    print(f"\n   Execute primeiro o notebook: aula-08-backtest-rigoroso")
    print(f"   Os dados devem ficar em: {DADOS_DIR}")
else:
    print("✓  Todos os arquivos necessários encontrados.")

In [ ]:
# Instalar o SDK da Anthropic (se necessário)
# !pip install anthropic

import anthropic
import os
import json
import textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# A API key deve estar na variável de ambiente ANTHROPIC_API_KEY
# Configure com: export ANTHROPIC_API_KEY="sk-ant-..."
# ou via arquivo .env + python-dotenv
client = anthropic.Anthropic()  # lê ANTHROPIC_API_KEY automaticamente

print('SDK Anthropic configurado.')
print(f'Versão: {anthropic.__version__}')

## 1. Como as LLMs Funcionam: A Arquitetura Transformer para Análise

As **LLMs (Large Language Models / Grandes Modelos de Linguagem)**, como o Claude que utilizamos, deixaram de ser meros brinquedos de chat para se tornarem assistentes analíticos extremamente potentes no fluxo quantitativo. Para usá-los com máxima eficiência, precisamos entender a intuição por trás do seu funcionamento para iniciantes.

---

### A Tarefa Fundamental: Previsão do Próximo Token
No nível mais básico, um LLM não "pensa" de forma consciente como um ser humano. Sua tarefa fundamental é puramente estatística: **dada uma sequência de texto de entrada (prompt), prever qual é a palavra (ou pedaço de palavra, chamado de token) mais provável de vir a seguir.**

Essa previsão não é simplória. Ela é alimentada por uma rede neural gigantesca treinada em trilhões de palavras da literatura humana, códigos de programação e relatórios financeiros.

---

### O Segredo: O Mecanismo de Atenção (Attention)
A tecnologia que permitiu o surgimento de LLMs modernos é a arquitetura **Transformer** (apresentada pelo Google em 2017 no artigo *"Attention Is All You Need"*).
O mecanismo de **Self-Attention (Auto-Atenção)** permite que o modelo decida, de forma dinâmica, em quais partes do texto de entrada ele deve focar para compreender o significado de uma palavra específica.

*Exemplo:* Na frase *"O fundo reduziu sua exposição no ativo devido à alta da volatilidade dele"*:
- Ao processar a palavra *"dele"*, o mecanismo de atenção consegue mapear matematicamente que ela se refere ao *"ativo"*, e não ao *"fundo"*. Isso permite que o modelo compreenda contextos longos, conexões lógicas sofisticadas e jargões do mercado financeiro com precisão milimétrica.

---

### Por que LLMs são fantásticos para Análise de Resultados?
LLMs são excelentes para tarefas de **síntese e raciocínio conceitual**:
- Eles conseguem ler uma tabela complexa de métricas líquidas (Sharpe, Drawdown, CAGR) e traduzir esses números em uma narrativa profissional impecável.
- Eles conseguem identificar incoerências lógicas em relatórios e sugerir críticas metodológicas construtivas baseadas nas melhores práticas acadêmicas de finanças.
- Eles aceleram o processo de rascunhar seções chatas do relatório de investimento, permitindo que os quants foquem no desenvolvimento do código e dos modelos estatísticos.


## 2. Onde a GenAI se Encaixa no Workflow Quantitativo (E onde NÃO deve entrar!)

A integração de Inteligência Artificial no mercado quantitativo requer um limite ético e metodológico rígido para evitar catástrofes de investimentos.

---

### O que DELEGAMOS para a GenAI (Aceleração e Qualidade)
1. **Rascunho de Código Pandas/Scipy:** Acelerar a escrita de funções complexas de otimização ou plotagem visual de gráficos.
2. **Narrativa e Redação Técnica:** Escrever a introdução conceitual da estratégia, explicar anomalias comportamentais ou estruturar o rascunho inicial do relatório final de investimentos.
3. **Análise de Cenários e Brainstorming de Riscos:** Pedir ao modelo para atuar como um "revisor severo da banca", listando as 5 principais fraquezas lógicas da nossa abordagem de momentum.

---

### O que NUNCA DELEGAMOS para a GenAI (O Papel do Humano e do Código)
1. **A Execução do Backtest:** Nós **nunca** pedimos para a IA "calcular os retornos na mente dela" ou simular o Sharpe de cabeça. LLMs são notoriamente ruins para fazer contas matemáticas precisas de precisão flutuante. A simulação histórica deve ser executada de forma **determinística** por código Python rigoroso. A IA deve apenas receber os resultados computados de forma estruturada para comentá-los.
2. **A Tomada de Decisão Final sobre Risco:** O design da estratégia, o gerenciamento dos limites operacionais e a validação do pipeline de dados são responsabilidade e juízo crítico intransferível do pesquisador quantitativo.


## 4. Engenharia de Prompt para Finanças Quantitativas: Técnicas Avançadas

Para obter análises verdadeiramente profundas e profissionais do Claude (evitando comentários superficiais e genéricos), precisamos aplicar técnicas estruturadas de **Prompt Engineering**.

---

### Os Quatro Princípios do Prompt Vencedor

#### 1. Contexto Rico antes do Pedido (Contextualization)
Forneça à IA uma persona clara de altíssimo nível (ex: *"Você é um Diretor de Pesquisa Quantitativa sênior de uma grande gestora de recursos internacional"*). Descreva as regras e o universo do projeto antes de fazer a solicitação.

#### 2. Poucas Amostras (Few-Shot Prompting)
Forneça exemplos do tipo de resposta que você espera obter. Se você quer que o modelo escreva uma análise crítica de riscos, dê a ele um pequeno parágrafo mostrando o nível de detalhe, tom de voz técnico e rigor acadêmico desejados.

#### 3. Restrições Rígidas (Negative Constraints)
Defina com clareza o que o modelo **não** deve fazer (ex: *"Nunca use adjetivos floreados ou superlativos excessivos como 'perfeito', 'infalível' ou 'garantido'. Mantenha um tom sóbrio, cético, humilde e profissional típico de quants"*).

#### 4. Cadeia de Pensamento (Chain of Thought — CoT)
Peça explicitamente para a IA "pensar passo a passo antes de responder". Isso força o modelo a alocar tokens de processamento para rascunhar o raciocínio estatístico em uma seção interna antes de cuspir o texto final, melhorando drasticamente a coerência lógica e eliminando alucinações matemáticas.


In [ ]:
# Carregando métricas reais calculadas nas aulas anteriores
ret_mensais  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
ret_ibov     = ret_mensais['BOVA11.SA']
ret_v1       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet')).squeeze()
ret_v2       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_sinal_v2.parquet')).squeeze()
ret_wf       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_walkforward_liquido.parquet')).squeeze()
pesos_v2     = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_sinal_v2.parquet'))

def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    r = retornos.dropna()
    excesso = r - rf_mensal
    n = len(r)
    cagr = (1 + r).prod() ** (12 / n) - 1
    vol  = r.std() * np.sqrt(12)
    sharpe = (excesso.mean() / excesso.std()) * np.sqrt(12) if excesso.std() > 0 else np.nan
    downside = excesso[excesso < 0].std() * np.sqrt(12)
    sortino  = (excesso.mean() * 12 / downside) if downside > 0 else np.nan
    cum = (1 + r).cumprod()
    dd  = cum / cum.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan
    alpha, beta = np.nan, np.nan
    if benchmark is not None:
        b = benchmark.reindex(r.index).dropna()
        r2 = r.reindex(b.index)
        if len(r2) > 10:
            beta, intercept, *_ = stats.linregress(b.values, r2.values)
            alpha = intercept * 12
    return dict(nome=nome, cagr=cagr, vol=vol, sharpe=sharpe, sortino=sortino,
                max_drawdown=mdd, calmar=calmar, alpha=alpha, beta=beta,
                n_meses=n, inicio=r.index[0], fim=r.index[-1])

# Calcular todas as métricas
m_v1   = calcular_metricas(ret_v1,  benchmark=ret_ibov, nome='Momentum v1 (sinal bruto)')
m_v2   = calcular_metricas(ret_v2,  benchmark=ret_ibov, nome='Momentum v2 (vol-adjusted)')
m_wf   = calcular_metricas(ret_wf,  benchmark=ret_ibov, nome='Walk-forward líquido')
m_ibov = calcular_metricas(ret_ibov, nome='IBOVESPA (BOVA11)')

turnover_medio = (pesos_v2.diff().abs().sum(axis=1) / 2).dropna().mean()

print('Métricas calculadas:')
for m in [m_v1, m_v2, m_wf, m_ibov]:
    print(f"  {m['nome']}: CAGR={m['cagr']:.1%}, Sharpe={m['sharpe']:.2f}, MDD={m['max_drawdown']:.1%}")

In [ ]:
# Princípio 2: System prompt para definir o papel e o tom

SYSTEM_ANALISTA_QUANT = """Você é um analista quantitativo sênior com 15 anos de experiência em 
gestão de portfólios e pesquisa de fatores. Você escreve relatórios técnicos precisos e concisos 
para gestores e júris de competições de finanças quantitativas. Seu estilo é direto, evita 
exageros, cita limitações metodológicas com honestidade, e usa terminologia técnica correta.
Escreva sempre em português brasileiro."""

# Serializar métricas para texto estruturado
def metricas_para_texto(m):
    return f"""
Estratégia: {m['nome']}
Período: {m['inicio'].strftime('%b/%Y')} a {m['fim'].strftime('%b/%Y')} ({m['n_meses']} meses)
CAGR: {m['cagr']:.1%}
Volatilidade anual: {m['vol']:.1%}
Sharpe Ratio: {m['sharpe']:.2f}
Sortino Ratio: {m['sortino']:.2f}
Máximo Drawdown: {m['max_drawdown']:.1%}
Calmar Ratio: {m['calmar']:.2f}
Alpha a.a. vs BOVA11: {m['alpha']:.1%}
Beta vs BOVA11: {m['beta']:.2f}""".strip()

# Prompt com contexto rico
prompt_analise = f"""Analise a performance das estratégias abaixo e escreva dois parágrafos:
1) O que os números dizem sobre a qualidade das estratégias
2) A diferença mais importante entre v1, v2 e walk-forward

===== ESTRATÉGIA V1 =====
{metricas_para_texto(m_v1)}

===== ESTRATÉGIA V2 (vol-adjusted) =====
{metricas_para_texto(m_v2)}

===== WALK-FORWARD LÍQUIDO DE CUSTOS (0.5% por turno) =====
{metricas_para_texto(m_wf)}

===== BENCHMARK: IBOVESPA =====
{metricas_para_texto(m_ibov)}

Turnover médio mensal da estratégia v2: {turnover_medio:.1%}
"""

analise_performance = chamar_claude(prompt_analise, system=SYSTEM_ANALISTA_QUANT, max_tokens=600)
print('=== ANÁLISE DE PERFORMANCE ===')
print(textwrap.fill(analise_performance, width=90))

### Princípio 3: Pedir formato específico

LLMs seguem instruções de formato muito bem. Sempre especifique:
- Quantos parágrafos
- Tom (técnico, executivo, didático)
- O que **não** incluir (evita divagações)
- Estrutura esperada (se quiser JSON, markdown, lista)

In [ ]:
# Exemplo: geração de JSON estruturado
prompt_json = f"""Dado o seguinte resumo de uma estratégia quantitativa, retorne um JSON com:
- "pontos_fortes": lista de até 3 pontos fortes observados nas métricas
- "pontos_fracos": lista de até 3 limitações ou riscos
- "recomendacoes": lista de até 2 melhorias sugeridas

Retorne SOMENTE o JSON, sem texto adicional.

Métricas da estratégia de momentum vol-adjusted (walk-forward, líquida de custos):
{metricas_para_texto(m_wf)}
Turnover médio: {turnover_medio:.1%}/mês
Número de ações no portfólio: top-10% do IBOVESPA (~7-8 ações)
Benchmark: BOVA11 (ETF IBOVESPA)
"""

resposta_json = chamar_claude(prompt_json, system=SYSTEM_ANALISTA_QUANT, max_tokens=600)

try:
    analise_estruturada = json.loads(resposta_json)
    print('=== ANÁLISE ESTRUTURADA (JSON) ===')
    print('\nPONTOS FORTES:')
    for p in analise_estruturada.get('pontos_fortes', []):
        print(f'  + {p}')
    print('\nPONTOS FRACOS / LIMITAÇÕES:')
    for p in analise_estruturada.get('pontos_fracos', []):
        print(f'  - {p}')
    print('\nRECOMENDAÇÕES:')
    for r in analise_estruturada.get('recomendacoes', []):
        print(f'  → {r}')
except json.JSONDecodeError:
    print('Resposta (texto):')
    print(resposta_json)

### Princípio 4: Chain of thought — "pense antes de responder"

Para análises complexas, pedir ao modelo que raciocine passo a passo melhora a qualidade da resposta final. Isso é especialmente útil quando você quer que o modelo **identifique contradições** ou **avalie premissas**.

In [ ]:
# Chain of thought: o modelo primeiro raciocina, depois conclui
prompt_cot = f"""Você é um avaliador experiente de estratégias quantitativas.

Antes de responder, pense em voz alta:
1) Que premissas essa estratégia faz que podem não se sustentar fora da amostra?
2) Que tipos de regime de mercado beneficiam ou prejudicam momentum?
3) Há inconsistências entre as métricas que merecem atenção?

Depois de raciocinar, conclua com uma avaliação final em 3-5 frases.

ESTRATÉGIA: Momentum cross-sectional no IBOVESPA
- Sinal: retorno acumulado 12-1 meses, normalizado pela vol. realizada de 63 dias
- Portfólio: top 10% das ações por sinal (long-only, equal weight)
- Rebalanceamento: mensal
{metricas_para_texto(m_wf)}
"""

resposta_cot = chamar_claude(prompt_cot, system=SYSTEM_ANALISTA_QUANT, max_tokens=800)
print('=== ANÁLISE COM CHAIN OF THOUGHT ===')
# Exibir com quebras de linha razoáveis
for linha in resposta_cot.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

---
## 5. Escrevendo seções do relatório

Vamos produzir as principais seções do relatório de entrega usando a API. A estrutura típica de um relatório de competição quant tem:

1. **Resumo Executivo** — 1 página, para quem tem 5 minutos
2. **Tese de Investimento** — por que momentum funciona
3. **Metodologia** — como construímos a estratégia
4. **Resultados e Métricas** — o que os números dizem
5. **Análise de Risco** — o que pode dar errado
6. **Conclusão** — o que recomendamos

In [ ]:
# Seção 1: Resumo Executivo

CONTEXTO_ESTRATEGIA = f"""
CONTEXTO:
- Competição: Desafio Quant AI da Itaú Asset Management 2025
- Equipe: estudantes de graduação em engenharia/ciências da computação
- Horizonte de investimento: médio prazo (mensal)
- Universo: ações do IBOVESPA (~77 tickers)
- Período de backtesting: 2015-2024 (10 anos)

ESTRATÉGIA:
- Fator: momentum cross-sectional
- Sinal v2: retorno acumulado 12-1 meses dividido pela vol. realizada de 63 dias
- Portfólio: long-only, top 10% por sinal, equal weight
- Rebalanceamento: mensal
- Validação: walk-forward (treino 48m, teste 12m)
- Custo modelado: 0.5% por turno sobre turnover realizado

MÉTRICAS WALK-FORWARD LÍQUIDAS:
{metricas_para_texto(m_wf)}
Turnover médio mensal: {turnover_medio:.1%}

BENCHMARK:
{metricas_para_texto(m_ibov)}
"""

prompt_exec = f"""{CONTEXTO_ESTRATEGIA}

Escreva o Resumo Executivo do relatório. Deve ter:
- 3-4 parágrafos
- Parágrafo 1: o que é a estratégia e qual a tese central
- Parágrafo 2: principais resultados (use os números reais acima)
- Parágrafo 3: rigor metodológico (mencione walk-forward e custos)
- Parágrafo 4: próximos passos / limitações reconhecidas
- Tom: técnico mas acessível a gestores de risco experientes
- NÃO invente métricas. Use SOMENTE os números fornecidos acima.
"""

resumo_executivo = chamar_claude(prompt_exec, system=SYSTEM_ANALISTA_QUANT, max_tokens=700)
print('=== RESUMO EXECUTIVO ===')
for linha in resumo_executivo.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

In [ ]:
# Seção 2: Metodologia

prompt_metodologia = f"""{CONTEXTO_ESTRATEGIA}

Escreva a seção de Metodologia do relatório. Estruture em subseções:

**2.1 Universo de Ativos**
Descreva a escolha do IBOVESPA como universo e os filtros de qualidade aplicados 
(cobertura mínima de 80%, filtro de liquidez).

**2.2 Construção do Sinal**
Explique o sinal 12-1 (citar Jegadeesh & Titman, 1993), o skip month, e a 
normalização pela volatilidade realizada de 63 dias. Inclua a fórmula em LaTeX.

**2.3 Construção do Portfólio**
Descreva o ranking cross-sectional, seleção do top 10%, equal weight, 
e rebalanceamento mensal.

**2.4 Validação**
Descreva o protocolo walk-forward e a modelagem de custos de transação.

Tom: acadêmico e preciso. Cada subseção deve ter 2-3 parágrafos.
NÃO invente referências além de Jegadeesh & Titman (1993) e DeMiguel et al. (2009).
"""

metodologia = chamar_claude(prompt_metodologia, system=SYSTEM_ANALISTA_QUANT, max_tokens=1200)
print('=== METODOLOGIA ===')
for linha in metodologia.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90) if not linha.startswith('**') else linha)
    else:
        print()

In [ ]:
# Seção 3: Análise de Risco

prompt_risco = f"""{CONTEXTO_ESTRATEGIA}

Escreva a seção de Análise de Risco. Comente sobre os seguintes riscos:

1. Risco de modelo: o que pode fazer o fator momentum deixar de funcionar?
   (crowding, regime shift, momentum crash pós-crise)

2. Risco de mercado: beta de {m_wf['beta']:.2f} implica exposição ao mercado brasileiro.
   Como isso se traduz em cenários extremos?

3. Risco de liquidez: com portfólio concentrado em 7-8 ações e turnover de {turnover_medio:.0%}/mês,
   qual o risco de impacto de mercado em ações menos líquidas do IBOV?

4. Survivorship bias: use o IBOVESPA constituintes atuais, que pode não representar
   fielmente o universo histórico disponível no início do período.

5. Risco regulatório: ações no Brasil têm regras específicas de curto e dividendos.
   Como isso afeta a estratégia long-only?

Para cada risco, mencione a magnitude estimada e mitigações adotadas ou possíveis.
Tom: honesto sobre limitações, sem ser alarmista.
"""

analise_risco = chamar_claude(prompt_risco, system=SYSTEM_ANALISTA_QUANT, max_tokens=900)
print('=== ANÁLISE DE RISCO ===')
for linha in analise_risco.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

---
## 6. Workflow completo: métricas → draft do relatório

Agora integramos tudo em uma pipeline completa que:
1. Coleta as métricas calculadas no Python
2. Formata o contexto para o LLM
3. Gera cada seção do relatório
4. Salva o draft em markdown

In [ ]:
def gerar_secao(titulo, instrucoes, contexto, max_tokens=800):
    """Gera uma seção do relatório com feedback visual."""
    print(f'Gerando: {titulo}...')
    prompt = f"{contexto}\n\n{instrucoes}"
    conteudo = chamar_claude(prompt, system=SYSTEM_ANALISTA_QUANT, max_tokens=max_tokens)
    return f"## {titulo}\n\n{conteudo}\n"

# Instrução para conclusão
instrucao_conclusao = """
Escreva a Conclusão do relatório. Deve:
- Retomar a tese central em 1 frase
- Afirmar os principais resultados quantitativos (use os números reais)
- Reconhecer as limitações mais importantes
- Propor 2-3 extensões naturais da pesquisa (ex: multi-fator, custo real, outros mercados)
- Finalizar com uma declaração sobre o potencial prático da estratégia
Máximo 3 parágrafos.
"""

conclusao = gerar_secao('Conclusão', instrucao_conclusao, CONTEXTO_ESTRATEGIA, max_tokens=500)
print('\n' + conclusao)

In [ ]:
# Montar o relatório completo e salvar

header_relatorio = f"""# Relatório de Estratégia Quantitativa
## Momentum Cross-Sectional no IBOVESPA

**Equipe:** ImpactUFSCar — Diretoria Quant  
**Competição:** Desafio Quant AI da Itaú Asset Management 2025  
**Data:** {pd.Timestamp.today().strftime('%B %Y')}  

---

"""

# Tabela de métricas formatada em markdown
def tabela_metricas_md(*dicts_metricas):
    linhas = []
    linhas.append('| Métrica | ' + ' | '.join(d['nome'] for d in dicts_metricas) + ' |')
    linhas.append('|---------|' + '|'.join('---------' for _ in dicts_metricas) + '|')
    campos = [
        ('CAGR', 'cagr', '{:.1%}'),
        ('Vol. Anual', 'vol', '{:.1%}'),
        ('Sharpe', 'sharpe', '{:.2f}'),
        ('Sortino', 'sortino', '{:.2f}'),
        ('Max Drawdown', 'max_drawdown', '{:.1%}'),
        ('Calmar', 'calmar', '{:.2f}'),
        ('Alpha a.a.', 'alpha', '{:.1%}'),
        ('Beta', 'beta', '{:.2f}'),
    ]
    for label, key, fmt in campos:
        vals = []
        for d in dicts_metricas:
            v = d.get(key, float('nan'))
            vals.append(fmt.format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A')
        linhas.append(f'| {label} | ' + ' | '.join(vals) + ' |')
    return '\n'.join(linhas)

tabela_md = tabela_metricas_md(m_v1, m_v2, m_wf, m_ibov)

relatorio_completo = (
    header_relatorio
    + '## 1. Resumo Executivo\n\n' + resumo_executivo + '\n\n'
    + '## 2. Metodologia\n\n' + metodologia + '\n\n'
    + '## 3. Resultados\n\n'
    + '### 3.1 Métricas de Performance\n\n'
    + tabela_md + '\n\n'
    + '### 3.2 Análise de Performance\n\n' + analise_performance + '\n\n'
    + '## 4. Análise de Risco\n\n' + analise_risco + '\n\n'
    + conclusao
)

# Salvar draft
with open('relatorio_draft.md', 'w', encoding='utf-8') as f:
    f.write(relatorio_completo)

print(f'Draft salvo: relatorio_draft.md')
print(f'Tamanho: {len(relatorio_completo):,} caracteres')
print(f'Palavras estimadas: {len(relatorio_completo.split()):,}')

---
## 7. Usando o Claude para crítica e revisão

Uma das aplicações mais poderosas: pedir ao modelo para **criticar** o próprio relatório.

In [ ]:
# Revisão crítica: o modelo avalia o relatório como se fosse um júri

SYSTEM_JURI = """Você é um júri experiente de uma competição de finanças quantitativas,
composto por gestores da Itaú Asset Management. Você avalia criticamente relatórios de 
estratégias quantitativas, identificando:
- Afirmações não suportadas por evidências
- Limitações metodológicas não reconhecidas pelo autor
- Métricas mal interpretadas
- Argumentos circulares ou fracos
Seja direto e construtivo."""

# Lemos o resumo executivo gerado
prompt_revisao = f"""Leia o resumo executivo abaixo e responda:

1. Quais as 3 afirmações mais fortes e bem fundamentadas?
2. Quais as 2-3 lacunas ou fraquezas que um avaliador exigente questionaria?
3. O que está faltando que uma estratégia realmente pronta para produção teria?

RESUMO EXECUTIVO:
{resumo_executivo}

Métricas de referência (para checar consistência):
{metricas_para_texto(m_wf)}
"""

critica = chamar_claude(prompt_revisao, system=SYSTEM_JURI, max_tokens=700)
print('=== CRÍTICA DO JÚRI ===')
for linha in critica.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

In [ ]:
# Conversa multi-turno: refinar uma seção com base na crítica
# O SDK suporta histórico de mensagens diretamente

def conversa_refinamento(secao_original, critica_recebida):
    """Refina uma seção com base em crítica usando conversa multi-turno."""
    mensagens = [
        {'role': 'user', 'content': f'Aqui está o resumo executivo que escrevi:\n\n{secao_original}'},
        {'role': 'assistant', 'content': 'Entendido. Li o resumo executivo.'},
        {'role': 'user', 'content': f'Um avaliador fez estas críticas:\n\n{critica_recebida}\n\n'
                                    'Reescreva o resumo incorporando as melhorias mais importantes. '
                                    'Mantenha o mesmo tamanho e os números originais.'}
    ]
    
    response = client.messages.create(
        model=MODELO,
        max_tokens=700,
        system=SYSTEM_ANALISTA_QUANT,
        messages=mensagens
    )
    return response.content[0].text

resumo_refinado = conversa_refinamento(resumo_executivo, critica)
print('=== RESUMO EXECUTIVO REFINADO ===')
for linha in resumo_refinado.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

---
## 8. Custo e limites da API

Antes de usar em produção, entenda os custos e limites:

## 9. Os Critérios de Avaliação do Desafio Itaú Asset e a Preparação do Pitch

Para encerrar o Intensivão Quant AI com chave de ouro, precisamos entender como o nosso projeto final (o Relatório Técnico de Investimento e a Apresentação Oral) será julgado pela banca de gestores e pesquisadores da **Itaú Asset Management**.

---

### Os 7 Critérios Fundamentais de Avaliação

#### 1. Rigor Metodológico (Ausência de Vieses)
A banca inspecionará detalhadamente se a equipe cometeu erros amadores de modelagem. A presença de **Look-Ahead Bias** ou a falta de tratamento adequado de **Survivorship Bias** derrubará a nota drasticamente. O uso de métricas como o **Deflated Sharpe Ratio (DSR)** demonstrará excelente rigor conceitual.

#### 2. Realismo Operacional (Fricções)
Estratégias puramente teóricas que fingem que custos de transação não existem serão duramente criticadas. A banca avaliará se a simulação incorpora custos de transação realistas (corretagem, slippage, emolumentos B3) e se o **Turnover** da carteira foi controlado de forma realista (por meio de bounds ou sinais normalizados por volatilidade).

#### 3. Alinhamento Teórico-Prático (Lógica Econômica)
Por que a estratégia funciona? A equipe precisa demonstrar clareza sobre os fatores econômicos e vieses comportamentais que sustentam o momentum e justificar metodologicamente escolhas cruciais como a janela 12-1 e a alocação restrita de Markowitz.

#### 4. Gestão de Risco e Consistência
A banca não busca simplesmente quem obteve o maior retorno bruto no histórico. A equipe vencedora demonstrará profundo entendimento do comportamento de risco da carteira, reportando de forma sóbria o **Max Drawdown**, a volatilidade e as limitações inerentes de "Momentum Crashes" da estratégia.

#### 5. Clareza da Narrativa Científica
O relatório final deve ser escrito de forma fluida, clara, concisa e altamente técnica. A "linha vermelha" (red line) do argumento deve conectar logicamente a introdução teórica, a análise exploratória, a construção de sinais, a simulação e os resultados líquidos.

#### 6. Qualidade dos Artefatos Visuais
A apresentação das tabelas de métricas e o **Tear Sheet** final devem ser impecáveis e esteticamente premium. Gráficos limpos, legíveis e profissionais facilitam a leitura rápida por tomadores de decisão.

#### 7. Defesa do Pitch Oral
Na sessão de perguntas e respostas com os jurados do Itaú, a equipe deve responder com precisão, calma e honestidade intelectual, sem tentar inventar desculpas para as limitações reais do modelo. Reconhecer as fraquezas da estratégia com sobriedade científica é um sinal de extrema maturidade quantitativa!


In [ ]:
# Diagrama de cobertura: o que cada aula entregou por dimensão avaliativa

criterios = [
    'Tese e fundamentação',
    'Qualidade dos dados',
    'Rigor metodológico',
    'Integridade do backtest',
    'Métricas e resultados',
    'Análise de risco',
    'Comunicação',
]

# Cobertura por aula (escala 0-1)
cobertura = {
    'Aula 1\n(Kickoff)':        [0.9, 0.0, 0.2, 0.2, 0.0, 0.1, 0.5],
    'Aula 2\n(Dados)':          [0.0, 0.9, 0.3, 0.4, 0.0, 0.2, 0.3],
    'Aula 3\n(EDA)':            [0.2, 1.0, 0.4, 0.3, 0.1, 0.5, 0.3],
    'Aula 4\n(Sinal v1)':       [0.8, 0.3, 0.7, 0.5, 0.4, 0.3, 0.4],
    'Aula 5\n(Backtest v1)':    [0.3, 0.2, 0.6, 0.7, 0.9, 0.5, 0.5],
    'Aula 6\n(Alocação)':       [0.5, 0.1, 0.8, 0.5, 0.7, 0.4, 0.5],
    'Aula 7\n(Sinal v2)':       [0.6, 0.2, 0.9, 0.7, 0.8, 0.5, 0.4],
    'Aula 8\n(Backtest rig.)':  [0.2, 0.3, 0.8, 1.0, 0.7, 0.9, 0.5],
    'Aula 9\n(GenAI)':          [0.3, 0.1, 0.4, 0.3, 0.5, 0.4, 1.0],
    'Aula 9\n(Defesa)':        [0.7, 0.2, 0.6, 0.5, 0.5, 0.7, 1.0],
}

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(criterios))
n_aulas = len(cobertura)
width = 0.08
colors = plt.cm.Blues(np.linspace(0.3, 0.9, n_aulas))

for i, (nome_aula, vals) in enumerate(cobertura.items()):
    offset = (i - n_aulas / 2) * width + width / 2
    bars = ax.bar(x + offset, vals, width * 0.9, label=nome_aula,
                  color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(criterios, fontsize=9)
ax.set_ylabel('Cobertura (0 → 1)')
ax.set_title('Mapeamento do intensivão para os critérios de avaliação')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('mapeamento_criterios.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 2. O argumento central — a "linha vermelha" do relatório

Todo relatório forte tem uma **linha vermelha**: um argumento central que conecta cada seção.

Para nossa estratégia, a linha vermelha é:

> *"Momentum cross-sectional no IBOVESPA, quando construído com fundamento econômico sólido, rigor metodológico e validação out-of-sample líquida de custos, gera alpha consistente — demonstrando que padrões comportamentais de underreaction persistem no mercado brasileiro."*

Cada seção do relatório deve **fortalecer** ou **qualificar honestamente** esse argumento:

```
Tese          → Por que momentum existe (Jegadeesh & Titman, 1993; underreaction)
Dados         → Que mercado estamos observando e quão limpos são os dados
Metodologia   → Como construímos o sinal — e por que essa escolha
Resultados    → Os números suportam a tese?
Risco         → Em que condições a tese falha?
Conclusão     → O argumento se sustenta, com as devidas qualificações
```

---
## 3. Estrutura do relatório final

A seguir, o template recomendado com indicação de tamanho e artefatos esperados em cada seção.

In [ ]:
# Template do relatório final
template_relatorio = [
    {
        'secao': 'Capa',
        'tamanho': '1 pág.',
        'conteudo': 'Título, equipe, instituição, data, resumo em 3 linhas',
        'artefatos': '—',
        'aula_origem': '—',
    },
    {
        'secao': '1. Resumo Executivo',
        'tamanho': '1 pág.',
        'conteudo': 'Tese, resultado-chave, método, limitação principal',
        'artefatos': 'Tabela de métricas simplificada',
        'aula_origem': 'Aula 9 (GenAI)',
    },
    {
        'secao': '2. Tese de Investimento',
        'tamanho': '1-2 pág.',
        'conteudo': 'Por que momentum funciona, evidências empíricas, mercado brasileiro',
        'artefatos': 'Gráfico de probabilidade condicional (Aula 4)',
        'aula_origem': 'Aulas 1 e 4',
    },
    {
        'secao': '3. Dados e Universo',
        'tamanho': '1 pág.',
        'conteudo': 'Fonte, período, filtros, NaNs, ajuste de dividendos',
        'artefatos': 'Tabela de cobertura por ticker, histograma de retornos',
        'aula_origem': 'Aulas 2 e 3',
    },
    {
        'secao': '4. Metodologia',
        'tamanho': '2-3 pág.',
        'conteudo': 'Sinal (fórmula), normalização, ranking, alocação, rebalanceamento',
        'artefatos': 'Linha do tempo do sinal (diagrama 12-1), equações em LaTeX',
        'aula_origem': 'Aulas 4, 6 e 7',
    },
    {
        'secao': '5. Resultados',
        'tamanho': '2-3 pág.',
        'conteudo': 'Todas as métricas (IS, WF, líquidas), tear sheet, heatmap',
        'artefatos': 'tearsheet_v1.png, heatmap mensal, tabela comparativa v1/v2/WF',
        'aula_origem': 'Aulas 5, 7 e 8',
    },
    {
        'secao': '6. Análise de Risco',
        'tamanho': '1-2 pág.',
        'conteudo': 'Survivorship bias, custos (break-even), regimes adversos a momentum',
        'artefatos': 'Gráfico break-even de custos, rolling Sharpe',
        'aula_origem': 'Aulas 3 e 8',
    },
    {
        'secao': '7. Conclusão',
        'tamanho': '0.5 pág.',
        'conteudo': 'Reafirma tese, reconhece limitações, propõe extensões',
        'artefatos': '—',
        'aula_origem': 'Aula 9 (GenAI)',
    },
    {
        'secao': '8. Referências',
        'tamanho': '0.5 pág.',
        'conteudo': 'J&T 1993, DeMiguel 2009, Bailey & López de Prado 2014',
        'artefatos': '—',
        'aula_origem': '—',
    },
]

df_template = pd.DataFrame(template_relatorio).set_index('secao')
print('=== TEMPLATE DO RELATÓRIO FINAL ===')
print(df_template[['tamanho', 'aula_origem']].to_string())
print(f'\nTotal estimado: ~10-12 páginas (excluindo capa e referências)')

---
## 4. O tear sheet final — a imagem mais importante do relatório

Gestores profissionais são acostumados a tear sheets. Uma boa tear sheet comunica em segundos o que a estratégia faz. Vamos produzir a versão final, combinando todos os resultados do intensivão.

In [ ]:
# Tear sheet final — comparação completa de todas as versões
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Estratégia de Momentum Cross-Sectional — IBOVESPA\nTear Sheet Final',
             fontsize=14, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)

# --- Painel 1: Retorno Cumulativo ---
ax1 = fig.add_subplot(gs[0, :])
idx_comum = ret_v1.dropna().index.intersection(ret_v2.dropna().index).intersection(ret_wf.dropna().index)

((1 + ret_v1.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='steelblue', lw=1.5, alpha=0.7, label='Sinal v1 (bruto, IS)')
((1 + ret_v2.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='royalblue', lw=2, label='Sinal v2 (vol-adj., IS)')
((1 + ret_wf.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='navy', lw=2.5, linestyle='--', label='Walk-forward líquido (OOS)')
((1 + ret_ibov.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='gray', lw=1.5, linestyle=':', label='IBOVESPA')
ax1.set_title('Retorno Cumulativo')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlabel('')

# --- Painel 2: Drawdown (walk-forward) ---
ax2 = fig.add_subplot(gs[1, :2])
cum_wf = (1 + ret_wf.reindex(idx_comum)).cumprod()
dd_wf  = cum_wf / cum_wf.cummax() - 1
ax2.fill_between(dd_wf.index, dd_wf.values, 0, color='tomato', alpha=0.6)
ax2.plot(dd_wf.index, dd_wf.values, color='darkred', lw=0.8)
ax2.set_title('Drawdown — Walk-forward líquido')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax2.set_xlabel('')

# --- Painel 3: Rolling Sharpe 24m ---
ax3 = fig.add_subplot(gs[1, 2])
rolling_sh = ret_wf.reindex(idx_comum).rolling(24).apply(
    lambda r: (r.mean()/r.std())*np.sqrt(12) if r.std() > 0 else np.nan)
rolling_sh.plot(ax=ax3, color='navy', lw=1.5)
ax3.axhline(0, color='black', lw=0.8)
ax3.axhline(1.0, color='green', lw=1, linestyle='--', alpha=0.7)
ax3.set_title('Rolling Sharpe 24m (OOS)')
ax3.set_xlabel('')

# --- Painel 4: Retorno anual ---
ax4 = fig.add_subplot(gs[2, :2])
ret_anual = ret_wf.reindex(idx_comum).resample('YE').apply(lambda r: (1+r).prod()-1)
ibov_anual = ret_ibov.reindex(idx_comum).resample('YE').apply(lambda r: (1+r).prod()-1)
anos = ret_anual.index.year
x = np.arange(len(anos))
ax4.bar(x - 0.2, ret_anual.values, 0.38,
        color=['steelblue' if v >= 0 else 'tomato' for v in ret_anual.values],
        label='Estratégia WF', alpha=0.85)
ax4.bar(x + 0.2, ibov_anual.values, 0.38,
        color=['lightgray' if v >= 0 else 'salmon' for v in ibov_anual.values],
        label='IBOVESPA', alpha=0.7)
ax4.set_xticks(x)
ax4.set_xticklabels(anos, rotation=45, fontsize=8)
ax4.axhline(0, color='black', lw=0.8)
ax4.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax4.set_title('Retorno Anual — WF vs IBOVESPA')
ax4.legend(fontsize=8)

# --- Painel 5: Tabela de métricas ---
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis('off')

metricas_tabela = [
    ['Métrica', 'Estratégia', 'IBOVESPA'],
    ['CAGR', f"{m_wf['cagr']:.1%}", f"{m_ibov['cagr']:.1%}"],
    ['Vol. Anual', f"{m_wf['vol']:.1%}", f"{m_ibov['vol']:.1%}"],
    ['Sharpe', f"{m_wf['sharpe']:.2f}", f"{m_ibov['sharpe']:.2f}"],
    ['Sortino', f"{m_wf['sortino']:.2f}", f"{m_ibov['sortino']:.2f}"],
    ['Max DD', f"{m_wf['max_drawdown']:.1%}", f"{m_ibov['max_drawdown']:.1%}"],
    ['Alpha a.a.', f"{m_wf['alpha']:.1%}", '—'],
    ['Beta', f"{m_wf['beta']:.2f}", '—'],
    ['Turnover/mês', f"{turnover_medio:.0%}", '—'],
]
tabela = ax5.table(cellText=metricas_tabela[1:], colLabels=metricas_tabela[0],
                   loc='center', cellLoc='center')
tabela.auto_set_font_size(False)
tabela.set_fontsize(8)
tabela.scale(1.0, 1.4)
for (row, col), cell in tabela.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c4a7c')
        cell.set_text_props(color='white', fontweight='bold')
    elif col == 1 and row > 0:
        cell.set_facecolor('#e8f0fe')
ax5.set_title('Walk-forward líquido\n(custo 0.5%/turno)', fontsize=8, pad=12)

plt.savefig('tearsheet_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tear sheet final salvo: tearsheet_final.png')

---
## 5. Defendendo escolhas metodológicas

### A estrutura de uma boa defesa

Para qualquer escolha metodológica, a defesa tem três camadas:

```
1. FUNDAMENTO      → Existe evidência teórica ou empírica para essa escolha?
2. TRADE-OFFS      → Quais alternativas foram consideradas e por que foram preteridas?
3. SENSIBILIDADE   → A estratégia funciona se esse parâmetro variar razoavelmente?
```

### Exemplos de perguntas e respostas

Vamos praticar as perguntas mais comuns do júri.

In [ ]:
# Compilando as questões mais comuns e estruturando respostas

defesa = [
    {
        'pergunta': 'Por que a janela 12-1 meses?',
        'fundamento': 'Jegadeesh & Titman (1993) demonstraram empiricamente que retornos entre '
                      '3 e 12 meses atrás têm poder preditivo positivo. O skip do mês mais recente '
                      'evita o efeito de reversão de curto prazo documentado na literatura.',
        'alternativas': '3-1, 6-1, 24-1 foram exploradas; 12-1 tem a maior base de evidências '
                        'internacionais e resultados consistentes na amostra brasileira.',
        'sensibilidade': 'Janelas entre 9 e 15 meses produzem resultados qualitativamente '
                         'similares, confirmando robustez.',
    },
    {
        'pergunta': 'Por que vol de 63 dias para normalizar?',
        'fundamento': 'Queremos capturar volatilidade recente sem ruído excessivo. '
                      '63 dias ≈ 1 trimestre — horizonte natural para revisão de risco em '
                      'gestão profissional. Shorter (21d) é ruidoso; longer (252d) reage tarde.',
        'alternativas': 'Testamos 21d, 63d e 252d. 63d melhor equilibra reatividade e estabilidade '
                        '(menor variância do Sharpe rolling).',
        'sensibilidade': 'Resultados são robustos para janelas entre 42 e 126 dias.',
    },
    {
        'pergunta': 'Por que top 10% e não top 20% ou top 5 ações?',
        'fundamento': 'Top 10% equilibra concentração de sinal (quanto maior o threshold, '
                      'mais puro o sinal) e diversificação (quanto menor, mais concentrado e '
                      'dependente de idiossincrasias individuais).',
        'alternativas': 'Top 5% é muito concentrado (3-4 ações) para um backtest mensal confiável. '
                        'Top 20% dilui o sinal com ações medianas no ranking.',
        'sensibilidade': 'Top 8-12% produz métricas semelhantes; a estratégia não é \'fragile\' '
                         'a essa escolha.',
    },
    {
        'pergunta': 'Por que equal weight e não signal-weighted?',
        'fundamento': 'DeMiguel et al. (2009) mostraram que equal weight supera portfólios '
                      'otimizados em dados reais em 14 das 14 métricas testadas, devido a '
                      'erros de estimação da covariância.',
        'alternativas': 'Testamos equal weight, signal-weighted e inverse-vol na Aula 6. '
                        'Equal weight obteve Sharpe mais consistente e menor turnover.',
        'sensibilidade': 'Signal-weighted entregou alpha similar com maior turnover (custo maior).',
    },
    {
        'pergunta': 'Os resultados são reais ou overfitting?',
        'fundamento': 'A janela 12-1 foi fixada antes de ver os dados, baseada em literatura '
                      'de 1993. Isso minimiza o número efetivo de testes para ~1.',
        'alternativas': 'Validação walk-forward (treino 48m, teste 12m) produz métricas '
                        'out-of-sample consistentes com in-sample. Degradação de '
                        f'{(m_wf["sharpe"]/m_v2["sharpe"]-1):.0%} é compatível com literatura.',
        'sensibilidade': 'Deflated Sharpe com ~1 teste efetivo: nossa estratégia é '
                         'estatisticamente significativa.',
    },
]

for item in defesa:
    print(f"\n{'='*70}")
    print(f"P: {item['pergunta']}")
    print(f"\n  Fundamento:")
    print(f"    {item['fundamento']}")
    print(f"\n  Alternativas consideradas:")
    print(f"    {item['alternativas']}")
    print(f"\n  Sensibilidade:")
    print(f"    {item['sensibilidade']}")

---
## 6. Falhas que contam como pontos

Um erro comum de equipes de primeira viagem: esconder limitações. O efeito é o oposto do esperado — o júri percebe, e a credibilidade da equipe despenca.

**Limitações bem documentadas demonstram maturidade metodológica.**

### Como reportar uma limitação corretamente

```
FRACO: "Nossa estratégia não usa survivorship bias."
         ↑ Vago, não quantifica, parece desculpa.

FORTE: "Utilizamos o constituinte atual do IBOVESPA como universo fixo, o que
         introduz survivorship bias positivo — estimamos que isso pode inflar o
         CAGR em 1-3% a.a. com base na literatura (Elton et al., 1996). Uma
         extensão natural é usar os constituintes históricos point-in-time."
         ↑ Quantifica, cita, propõe solução. Isso é maturidade.
```

In [ ]:
# Tabela de limitações — o que incluir no relatório

limitacoes = pd.DataFrame([
    {
        'Limitação': 'Survivorship bias',
        'Impacto estimado': '+1-3% CAGR a.a.',
        'Direção': 'Infla resultados',
        'Mitigação adotada': 'Filtro de liquidez p20 reduz ações marginais',
        'Extensão proposta': 'Usar constituintes históricos (Economatica/Bloomberg)',
    },
    {
        'Limitação': 'Dados não point-in-time',
        'Impacto estimado': 'Desconhecido',
        'Direção': 'Potencialmente infla',
        'Mitigação adotada': 'Preços de fechamento ajustados — impacto reduzido para momentum puro',
        'Extensão proposta': 'Provider com dados P.I.T. (ex: Refinitiv)',
    },
    {
        'Limitação': 'Custos modelados (não reais)',
        'Impacto estimado': '±0.5 Sharpe dependendo do custo real',
        'Direção': 'Ambígua',
        'Mitigação adotada': 'Análise de break-even: estratégia positiva até ~X% por turno',
        'Extensão proposta': 'Paper trading com custo real observado',
    },
    {
        'Limitação': 'Concentração (7-8 ações)',
        'Impacto estimado': 'Risco idiossincrático elevado',
        'Direção': 'Risco bilateral',
        'Mitigação adotada': 'Equal weight diversifica dentro do portfólio',
        'Extensão proposta': 'Ampliar para top 15-20% ou adicionar restrição de setor',
    },
    {
        'Limitação': 'Escopo único de fator',
        'Impacto estimado': 'Vulnerável a regimes anti-momentum',
        'Direção': 'Risco de crise (momentum crash)',
        'Mitigação adotada': 'Volatility adjustment (sinal v2) mitiga parcialmente',
        'Extensão proposta': 'Combinar momentum com value ou quality (multi-fator)',
    },
])

print('=== LIMITAÇÕES A INCLUIR NO RELATÓRIO ===')
print(limitacoes[['Limitação', 'Impacto estimado', 'Direção', 'Extensão proposta']].to_string(index=False))

---
## 7. Sessão de cross-review entre equipes

O cross-review é a parte final do intensivão. Cada equipe atua como **júri** das outras.

### Por que fazer isso

1. Aprender com variações metodológicas das outras equipes
2. Praticar a defesa sob pressão antes do dia real
3. Identificar pontos cegos que você não veria sozinho
4. Desenvolver a capacidade de avaliar estratégias quantitativas

### Protocolo da sessão

In [ ]:
# Rubrica de avaliação para cross-review

rubrica = pd.DataFrame([
    {'Critério': 'Clareza da tese',
     'Pontos': 10,
     'Pergunta-guia': 'Em uma frase, qual a hipótese central e está bem fundamentada?'},
    {'Critério': 'Rigor dos dados',
     'Pontos': 15,
     'Pergunta-guia': 'Os filtros foram aplicados? Vieses identificados e quantificados?'},
    {'Critério': 'Justificativa dos parâmetros',
     'Pontos': 20,
     'Pergunta-guia': 'Cada escolha tem fundamento econômico ou foi simplesmente otimizada?'},
    {'Critério': 'Integridade do backtest',
     'Pontos': 20,
     'Pergunta-guia': 'Há look-ahead? Walk-forward? Custos modelados?'},
    {'Critério': 'Qualidade dos resultados',
     'Pontos': 15,
     'Pergunta-guia': 'As métricas são plausíveis? Sharpe muito alto (>3) levanta suspeitas.'},
    {'Critério': 'Análise de risco',
     'Pontos': 10,
     'Pergunta-guia': 'As limitações foram reconhecidas honestamente?'},
    {'Critério': 'Comunicação e clareza',
     'Pontos': 10,
     'Pergunta-guia': 'Alguém sem ver o código entende a estratégia?'},
])

print(f'=== RUBRICA DO CROSS-REVIEW ===')
print(rubrica.to_string(index=False))
print(f'\nTotal: {rubrica["Pontos"].sum()} pontos')

In [ ]:
# Visualização da rubrica em gráfico de rosca
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de pizza — distribuição dos pontos
ax = axes[0]
cores = ['#1f4e79', '#2e75b6', '#4472c4', '#5b9bd5', '#70ad47', '#ffc000', '#ff0000']
wedges, texts, autotexts = ax.pie(
    rubrica['Pontos'],
    labels=[f"{r['Critério']}\n({r['Pontos']}pts)" for _, r in rubrica.iterrows()],
    colors=cores,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
)
for text in texts:
    text.set_fontsize(8)
ax.set_title('Distribuição de pontos\nna rubrica de cross-review', fontweight='bold')

# Timeline da sessão de cross-review
ax = axes[1]
ax.axis('off')

etapas = [
    ('00:00', 'Apresentação (10 min)', '#1f4e79'),
    ('00:10', 'Perguntas do júri (15 min)', '#c00000'),
    ('00:25', 'Deliberação interna (5 min)', '#7f7f7f'),
    ('00:30', 'Feedback estruturado (10 min)', '#375623'),
    ('00:40', 'Troca de papéis: próxima equipe', '#4472c4'),
]

for i, (tempo, descricao, cor) in enumerate(etapas):
    y = 0.85 - i * 0.18
    ax.add_patch(mpatches.FancyBboxPatch((0.05, y - 0.06), 0.9, 0.12,
                                          boxstyle='round,pad=0.02', facecolor=cor, alpha=0.15,
                                          edgecolor=cor, linewidth=2))
    ax.text(0.12, y, tempo, ha='left', va='center', fontsize=10,
            fontweight='bold', color=cor)
    ax.text(0.30, y, descricao, ha='left', va='center', fontsize=10)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Timeline da sessão de cross-review\n(por equipe avaliada)', fontweight='bold')

plt.tight_layout()
plt.savefig('crossreview_rubrica.png', dpi=120, bbox_inches='tight')
plt.show()

### As 5 perguntas que o júri sempre faz

Durante o cross-review, concentre as perguntas nas áreas com maior sinal-ruído:

1. **"Como vocês escolheram o parâmetro X?"**  
   → Procure por: "testamos vários e ficamos com o melhor" (red flag de overfitting)

2. **"Qual é a explicação econômica para essa estratégia funcionar?"**  
   → Procure por: ausência de hipótese comportamental ou estrutural

3. **"O que acontece com a estratégia em março de 2020?"** (COVID crash)  
   → Momentum sofre em reversões abruptas. Eles sabem disso?

4. **"Qual é o Sharpe out-of-sample?"**  
   → Se só reportam in-sample, pergunta crítica.

5. **"Se eu aumentar o custo de transação para 1%, a estratégia ainda funciona?"**  
   → Testa robustez a premissas de custo.

---
## 8. Checklist final de entrega

In [ ]:
checklist_final = [
    # Código e dados
    ('CÓDIGO', 'Notebooks numerados e funcionando do início ao fim sem erros'),
    ('CÓDIGO', 'Parâmetros principais em variáveis nomeadas (sem magic numbers)'),
    ('CÓDIGO', 'Todos os parquets gerados e presentes na pasta /dados'),
    ('CÓDIGO', 'Nenhum look-ahead bias: shift(2) no sinal, shift(1) nos pesos'),
    ('CÓDIGO', 'requirements.txt ou environment.yml com versões fixadas'),
    # Metodologia
    ('MÉTODO', 'Janela de momentum justificada com referência (J&T 1993)'),
    ('MÉTODO', 'Volatilidade rolling: janela escolhida com fundamento documentado'),
    ('MÉTODO', 'Alocação: por que equal weight (DeMiguel 2009 citado)'),
    ('MÉTODO', 'Turnover calculado e reportado'),
    # Backtest
    ('BACKTEST', 'Walk-forward: resultado out-of-sample presente'),
    ('BACKTEST', 'Custos de transação modelados (premissa explícita)'),
    ('BACKTEST', 'Análise de break-even de custos incluída'),
    ('BACKTEST', 'Survivorship bias reconhecido e quantificado aproximadamente'),
    # Relatório
    ('RELATÓRIO', 'Resumo executivo: 1 página com tese + resultado principal + limitação'),
    ('RELATÓRIO', 'Todas as métricas: CAGR, vol, Sharpe, Sortino, MDD, alpha, beta'),
    ('RELATÓRIO', 'Tear sheet incluída como figura'),
    ('RELATÓRIO', 'Seção de análise de risco com limitações documentadas'),
    ('RELATÓRIO', 'Referências: J&T 1993, DeMiguel 2009, Bailey & LdP 2014'),
    # Apresentação
    ('DEFESA', 'Cada escolha metodológica: fundamento → alternativas → sensibilidade'),
    ('DEFESA', 'Respostas para: "por que essa janela?", "por que equal weight?"'),
    ('DEFESA', 'Simulado de cross-review realizado com outra equipe'),
    ('DEFESA', 'Slides têm no máximo 15 palavras por slide'),
]

df_checklist = pd.DataFrame(checklist_final, columns=['Categoria', 'Item'])

print('=== CHECKLIST FINAL DE ENTREGA ===')
cat_atual = ''
for _, row in df_checklist.iterrows():
    if row['Categoria'] != cat_atual:
        cat_atual = row['Categoria']
        print(f"\n{'─'*50}")
        print(f" {cat_atual}")
        print(f"{'─'*50}")
    print(f"  [ ] {row['Item']}")

print(f"\nTotal de itens: {len(df_checklist)}")

---
## 9. O que separa os finalistas

Ao avaliar centenas de equipes em competições desse tipo, gestores experientes relatam que os finalistas têm em comum:

### Critérios de differenciação

| Aspecto | Maioria das equipes | Finalistas |
|---------|---------------------|------------|
| **Parâmetros** | "Testamos vários e ficamos com o melhor" | "Fixamos pela literatura antes de ver os dados" |
| **Limitações** | Ignoradas ou minimizadas | Quantificadas e com extensões propostas |
| **Custos** | Ignorados | Modelados com análise de sensibilidade |
| **Validação** | In-sample apenas | Walk-forward OOS explícito |
| **Sharpe alto** | Comemoram sem questionar | Investigam se há look-ahead ou overfitting |
| **Defesa** | Defensiva quando questionados | Antecipam críticas no relatório |
| **Comunicação** | Código no slide | Gráficos que contam uma história |

### A frase que resume tudo

> *"Não precisamos de uma estratégia perfeita. Precisamos de uma estratégia honesta, fundamentada, e bem comunicada — e da capacidade de defender cada escolha com evidência."*

In [ ]:
# Visão geral do intensivão: o que construímos

print('=== INTENSIVÃO QUANT AI — RESUMO DO QUE CONSTRUÍMOS ===')
print()
entregas = [
    ('Aula 1', 'Kickoff', 'Alinhamento sobre o que é uma estratégia quant e os critérios de avaliação'),
    ('Aula 2', 'Dados', 'Pipeline de coleta: ~77 tickers IBOV, retornos log, parquets'),
    ('Aula 3', 'EDA', 'Estatísticas descritivas, fat tails, ADF, ACF, filtros de qualidade'),
    ('Aula 4', 'Sinal v1', 'Momentum 12-1, ranking cross-sectional, turnover, primeira carteira'),
    ('Aula 5', 'Backtest v1', 'CAGR, Sharpe, Sortino, drawdown, Calmar, alpha/beta, tear sheet'),
    ('Aula 6', 'Alocação', 'Fronteira eficiente, equal vs signal vs inv-vol, DeMiguel 2009'),
    ('Aula 7', 'Sinal v2', 'Normalização por vol realizada 63d, comparação v1 vs v2'),
    ('Aula 8', 'Backtest rig.', 'Look-ahead, overfitting, multiple testing, DSR, custos, walk-forward'),
    ('Aula 9', 'GenAI', 'API Claude: resumo executivo, metodologia, análise de risco, draft'),
    ('Aula 9', 'GenAI/Rel', 'Tear sheet final, rubrica de cross-review, checklist de entrega'),
]
for aula, nome, entrega in entregas:
    print(f'  {aula:7s} — {nome:<18s} → {entrega}')

print()
print('Arquivos gerados:')
print('  dados/precos_ibov.parquet')
print('  dados/retornos_diarios_limpo.parquet')
print('  dados/retornos_mensais_limpo.parquet')
print('  dados/sinal_v1.parquet, sinal_v2.parquet')
print('  dados/pesos_v1.parquet, pesos_sinal_v2.parquet')
print('  dados/retorno_carteira_v1.parquet, ..._v2.parquet, ..._walkforward_liquido.parquet')
print('  aula-05-backtest-v1/tearsheet_v1.png')
print('  aula-09-genai-analise/relatorio_draft.md')
print('  aula-10-relatorio-defesa/tearsheet_final.png')

---
## Conclusão do Intensivão Quant AI

Parabéns! Você concluiu as 9 aulas do Intensivão Quant AI da ImpactUFSCar.
Você agora domina desde a coleta robusta de preços, estatística de séries temporais, construção de sinais ajustados por risco (vol-adjusted momentum), alocação com otimização (Markowitz), simulação rigorosa de custos, até a redação com suporte de LLMs e a estruturação de uma defesa oral premium.

O robô de vocês agora tem pernas — cabe a cada equipe refinar e torná-lo ultracompetitivo até a data de entrega final em 17/08/2026. Boa sorte no Desafio Itaú Quant AI!